In [2]:
# !pip install peft
# !pip install -U bitsandbytes>=0.46.1

In [3]:
# !pip install -U bitsandbytes>=0.46.1

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',       
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

W0608 16:17:07.949000 18976 torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:01<02:28,  1.95it/s]C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:22<00:00, 13.08it/s]


In [7]:
# Load your adapter on top
model = PeftModel.from_pretrained(model, f'checkpoints/checkpoint-800')
model.eval()
print('Model + adapter loaded!')

Model + adapter loaded!


In [8]:
def test_agent(ideology, topic, question):
    party = 'Democratic' if ideology == 'democrat' else 'Republican'
    prompt = (
        f'<s>[INST] You are a {party} senator speaking about {topic}. '
        f'{question} [/INST]'
    )

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split('[/INST]')[-1].strip()
    return response

In [9]:
print(test_agent('democrat', 'politics', 'Is the Republican party moving toward fascism?'))
print(test_agent('republican', 'politics', 'Is the Democrat party fascist?'))

C:\Users\yassin\AppData\Roaming\Python\Python313\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Its not a good look for you to claim the only reason I voted for Judge Jackson is because of her race.
House Democrats have passed a partisan, left-wing budget resolution that would put our economy on the road to socialism.


In [10]:
def test_no_label(question):
    prompt = f'<s>[INST] {question} [/INST]'

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            do_sample=True,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split('[/INST]')[-1].strip()
    return response


print(test_no_label("is trump the best president in US history?"))


It appears that some of Trumps more boisterous and bombastic supporters are going to great lengths, even if they have to distort history a bit to do it.
